# HOG + LBP + SVM ile Göz Bölgesi Üzerinden Deepfake Tespiti

Bu notebook, **Combined Eye ROI** görüntülerinden klasik makine öğrenmesi tabanlı bir baseline üretir.

Akış:
1. Google Drive ve merkezi YAML konfigürasyonu
2. Eye ROI metadata doğrulama
3. Veri muhasebesi ve split leakage kontrolleri
4. Combined Eye ROI görüntülerinde grayscale + resize
5. HOG + LBP özellik çıkarımı ve füzyonu
6. Train-only StandardScaler
7. Validation-only SVM + threshold seçimi
8. Atomik checkpoint ve fresh-load inference kalite kapıları
9. Frame-level ve grup/video-level nihai test
10. İngilizce, yüksek çözünürlüklü rapor grafikleri
11. Environment lock, manifest ve run summary

**Pozitif sınıf: Fake**  
**Seed: 42**  
**Pretrained model: kullanılmıyor**

> Not: Eye ROI metadata içindeki `video_id` alanı, mevcut veri üretiminde split/sınıf gruplarını (`real_train`, `fake_test` vb.) temsil ediyor. Notebook bunu uydurmaz. Bu nedenle `video_id` ile group-level değerlendirme üretilebilir; fakat bu alan gerçek orijinal video kimliği değilse bilimsel olarak "true video-level" genelleme metriği olarak yorumlanmamalıdır.


In [1]:

# 1) Colab bağımlılıkları
# İlk çalıştırmada güncel paketler kurulur; deney sonunda gerçek ortam `requirements_lock.txt` ile kilitlenir.
!pip -q install scikit-image scikit-learn pyyaml joblib opencv-python-headless


In [2]:
# 2) Google Drive'ı bağla ve merkezi YAML konfigürasyonunu oluştur/yükle
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import yaml

CONFIG_PATH = Path('/content/hog_lbp_svm_eye.yaml')

DEFAULT_CONFIG_YAML = '''seed: 42
region: eye
roi_variant: combined
model_name: hog_lbp_svm

image_width: 128
image_height: 64

hog:
  orientations: 9
  pixels_per_cell: [8, 8]
  cells_per_block: [2, 2]
  block_norm: L2-Hys

lbp:
  points: 8
  radius: 1
  method: uniform

svm:
  class_weight: balanced
  selection_metric: f1
  kernels: [linear, rbf]
  c_values: [0.1, 1.0, 10.0]
  gamma_values: [scale, 0.001, 0.0001]

positive_class: fake
figure_dpi: 150
minimum_figure_short_edge_px: 600
resume_run_id: null

experiment_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
data_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
roi_metadata_path: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
results_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar
'''

def load_or_repair_yaml(config_path: Path, default_text: str):
    if not config_path.exists():
        config_path.write_text(default_text, encoding='utf-8')

    try:
        with config_path.open('r', encoding='utf-8') as f:
            cfg = yaml.safe_load(f)
    except yaml.YAMLError:
        backup_path = config_path.with_suffix('.yaml.broken')
        if backup_path.exists():
            backup_path.unlink()
        config_path.replace(backup_path)
        config_path.write_text(default_text, encoding='utf-8')
        with config_path.open('r', encoding='utf-8') as f:
            cfg = yaml.safe_load(f)
        print(f'Bozuk YAML config yenilendi. Eski dosya: {backup_path}')

    if not isinstance(cfg, dict):
        raise ValueError('Config YAML bir sözlük/dict üretmedi.')

    return cfg

CONFIG = load_or_repair_yaml(CONFIG_PATH, DEFAULT_CONFIG_YAML)

required_config_keys = {
    'seed', 'region', 'roi_variant', 'model_name',
    'image_width', 'image_height',
    'hog', 'lbp', 'svm',
    'positive_class', 'figure_dpi',
    'minimum_figure_short_edge_px',
    'experiment_root', 'data_root',
    'roi_metadata_path', 'results_root',
}
missing_cfg = required_config_keys - set(CONFIG)
if missing_cfg:
    raise KeyError(f'Config eksik anahtarlar: {sorted(missing_cfg)}')

print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))


Mounted at /content/drive
seed: 42
region: eye
roi_variant: combined
model_name: hog_lbp_svm
image_width: 128
image_height: 64
hog:
  orientations: 9
  pixels_per_cell:
  - 8
  - 8
  cells_per_block:
  - 2
  - 2
  block_norm: L2-Hys
lbp:
  points: 8
  radius: 1
  method: uniform
svm:
  class_weight: balanced
  selection_metric: f1
  kernels:
  - linear
  - rbf
  c_values:
  - 0.1
  - 1.0
  - 10.0
  gamma_values:
  - scale
  - 0.001
  - 0.0001
positive_class: fake
figure_dpi: 150
minimum_figure_short_edge_px: 600
resume_run_id: null
experiment_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney
  1
data_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
roi_metadata_path: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney
  1/Göz/eye_roi_output/metadata.csv
results_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney
  1/Sonuçlar



In [3]:

# 3) Imports, reproducibility, run klasörleri ve yardımcı fonksiyonlar
import os
import sys
import json
import math
import time
import random
import hashlib
import logging
import platform
import subprocess
from datetime import datetime
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog, local_binary_pattern

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay,
)

SEED = int(CONFIG['seed'])
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

EXPERIMENT_ROOT = Path(CONFIG['experiment_root'])
DATA_ROOT = Path(CONFIG['data_root'])
ROI_METADATA_PATH = Path(CONFIG['roi_metadata_path'])
RESULTS_ROOT = Path(CONFIG['results_root'])

resume_run_id = CONFIG.get('resume_run_id')
if resume_run_id:
    RUN_ID = str(resume_run_id)
else:
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M') + f"_eye_hog_lbp_svm_seed{SEED}"

RUN_DIR = RESULTS_ROOT / RUN_ID
if RUN_DIR.exists() and not resume_run_id:
    raise FileExistsError(
        f'Bu run_id zaten mevcut: {RUN_DIR}. Aynı dizine iki farklı deney yazılmaz. '
        'Yeni bir dakika bekleyin veya resume_run_id ile bilinçli olarak devam edin.'
    )
DIRS = {
    'checkpoints': RUN_DIR / 'checkpoints',
    'logs': RUN_DIR / 'logs',
    'metrics': RUN_DIR / 'metrics',
    'predictions': RUN_DIR / 'predictions',
    'figures': RUN_DIR / 'figures',
    'artifacts': RUN_DIR / 'artifacts',
}
for path in [RUN_DIR, *DIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)

# Kaynak veriye yazılmadığını garanti eden temel yol kontrolü
assert DATA_ROOT.exists(), f'Data root bulunamadı: {DATA_ROOT}'
assert ROI_METADATA_PATH.exists(), f'ROI metadata bulunamadı: {ROI_METADATA_PATH}'
assert EXPERIMENT_ROOT.exists(), f'Deney 1 klasörü bulunamadı: {EXPERIMENT_ROOT}'
assert RESULTS_ROOT.exists(), f'Sonuçlar klasörü bulunamadı: {RESULTS_ROOT}'
assert RESULTS_ROOT.resolve().parent == EXPERIMENT_ROOT.resolve(), (
    f'Sonuçlar klasörü Deney 1 altında değil: {RESULTS_ROOT}'
)
assert RUN_DIR.resolve() != DATA_ROOT.resolve()
assert DATA_ROOT.resolve() not in RUN_DIR.resolve().parents
assert RUN_DIR.resolve().parent == RESULTS_ROOT.resolve()

# Logger
LOG_PATH = DIRS['logs'] / 'run.log'
logger = logging.getLogger('eye_hog_lbp_svm')
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler = logging.FileHandler(LOG_PATH, encoding='utf-8')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json_atomic(data, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    with temp.open('w', encoding='utf-8') as f:
        json.dump(json_safe(data), f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    # Doğrulama
    with temp.open('r', encoding='utf-8') as f:
        json.load(f)
    os.replace(temp, target)


def write_csv_atomic(df: pd.DataFrame, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    df.to_csv(temp, index=False)
    _ = pd.read_csv(temp)
    os.replace(temp, target)


def save_npz_atomic(target: Path, **arrays):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    with temp.open('wb') as f:
        np.savez_compressed(f, **arrays)
        f.flush()
        os.fsync(f.fileno())
    with np.load(temp, allow_pickle=False) as z:
        if not z.files:
            raise RuntimeError(f'NPZ doğrulama başarısız: {temp}')
    os.replace(temp, target)


def atomic_joblib_dump(obj, target: Path, required_keys=None):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    joblib.dump(obj, temp)
    loaded = joblib.load(temp)
    if required_keys:
        if not isinstance(loaded, dict) or not set(required_keys).issubset(loaded.keys()):
            raise RuntimeError(f'Checkpoint doğrulama başarısız: {temp}')
    os.replace(temp, target)


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def save_figure(fig, target: Path):
    fig.tight_layout()
    fig.savefig(target, dpi=int(CONFIG['figure_dpi']), bbox_inches='tight')
    plt.close(fig)
    with Image.open(target) as img:
        width, height = img.size
    min_edge = int(CONFIG['minimum_figure_short_edge_px'])
    assert min(width, height) >= min_edge, f'Figure resolution too low: {(width, height)}'
    return width, height


resolved_config = dict(CONFIG)
resolved_config.update({
    'run_id': RUN_ID,
    'run_dir': str(RUN_DIR),
    'created_at': datetime.now().isoformat(),
})
with (RUN_DIR / 'config_resolved.yaml').open('w', encoding='utf-8') as f:
    yaml.safe_dump(resolved_config, f, sort_keys=False, allow_unicode=True)

logger.info('RUN_ID=%s', RUN_ID)
logger.info('Input=%s', DATA_ROOT)
logger.info('Output=%s', RUN_DIR)


2026-08-07 11:19:46,600 | INFO | RUN_ID=20260807_1119_eye_hog_lbp_svm_seed42


INFO:eye_hog_lbp_svm:RUN_ID=20260807_1119_eye_hog_lbp_svm_seed42


2026-08-07 11:19:46,632 | INFO | Input=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output


INFO:eye_hog_lbp_svm:Input=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output


2026-08-07 11:19:46,636 | INFO | Output=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1119_eye_hog_lbp_svm_seed42


INFO:eye_hog_lbp_svm:Output=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1119_eye_hog_lbp_svm_seed42


In [6]:
# ============================================================
# CELL 4 — EYE ROI METADATA + DATA ACCOUNTING
#          + SPLIT / LEAKAGE QUALITY GATES
# ============================================================

roi = pd.read_csv(
    ROI_METADATA_PATH
)
import re

# ============================================================
# 1. REQUIRED ROI METADATA SCHEMA
# ============================================================

required_roi_cols = {
    "sample_id",
    "source_frame",
    "relative_frame_path",
    "label",
    "split",
    "video_id",
    "frame_stem",
    "face_id",
    "combined_eye_path",
    "status",
}


missing_roi = (
    required_roi_cols
    -
    set(roi.columns)
)


if missing_roi:

    raise AssertionError(
        "Eye ROI metadata eksik sütunlar: "
        f"{sorted(missing_roi)}"
    )


roi = roi.copy()


# ============================================================
# 2. SAFE STRING NORMALIZATION
# ============================================================
#
# ÖNEMLİ:
# astype(str) kullanmadan önce NaN değerlerini koruyoruz.
#
# Çünkü:
#     NaN -> "nan"
#
# dönüşümü bütün no_face satırlarının aynı sample_id'ye
# sahipmiş gibi görünmesine neden olur.
# ============================================================

def normalize_optional_string(series):

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


roi["sample_id"] = normalize_optional_string(
    roi["sample_id"]
)

roi["source_frame"] = normalize_optional_string(
    roi["source_frame"]
)

roi["relative_frame_path"] = normalize_optional_string(
    roi["relative_frame_path"]
)

roi["label"] = (
    normalize_optional_string(
        roi["label"]
    )
    .str.lower()
)

roi["split"] = (
    normalize_optional_string(
        roi["split"]
    )
    .str.lower()
)

roi["status_raw"] = (
    normalize_optional_string(
        roi["status"]
    )
    .str.lower()
)

roi["video_id"] = normalize_optional_string(
    roi["video_id"]
)

roi["frame_stem"] = normalize_optional_string(
    roi["frame_stem"]
)


# ============================================================
# 3. LABEL / SPLIT QUALITY GATES
# ============================================================

found_labels = set(
    roi["label"].unique()
)


if found_labels != {
    "real",
    "fake",
}:

    raise AssertionError(
        "Beklenmeyen label değerleri: "
        f"{roi['label'].value_counts(dropna=False).to_dict()}"
    )


found_splits = set(
    roi["split"].unique()
)


if found_splits != {
    "train",
    "val",
    "test",
}:

    raise AssertionError(
        "Beklenmeyen split değerleri: "
        f"{roi['split'].value_counts(dropna=False).to_dict()}"
    )


# ============================================================
# 4. STATUS STANDARDIZATION
# ============================================================
#
# Gerçek Eye ROI metadata:
#
#     ok
#     no_face
#
# biçimindedir.
# ============================================================

status_map = {

    "ok": "SUCCESS",

    "success": "SUCCESS",

    "no_face": "SKIPPED",

    "skipped": "SKIPPED",

    "error": "ERROR",
}


unknown_statuses = (
    set(
        roi["status_raw"].unique()
    )
    -
    set(status_map)
)


if unknown_statuses:

    raise AssertionError(
        "Bilinmeyen Eye ROI status değerleri: "
        f"{sorted(unknown_statuses)}"
    )


roi["status_std"] = (
    roi["status_raw"]
    .map(status_map)
)


# ============================================================
# 5. SAMPLE-ID VALIDATION
# ============================================================
#
# Kritik düzeltme:
#
# no_face satırlarında ROI üretilmediği için gerçek sample_id
# bulunmayabilir.
#
# Bu nedenle sample_id zorunluluğunu yalnızca SUCCESS satırlarına
# uyguluyoruz.
# ============================================================

success_roi_mask = (
    roi["status_std"]
    ==
    "SUCCESS"
)


skipped_roi_mask = (
    roi["status_std"]
    ==
    "SKIPPED"
)


error_roi_mask = (
    roi["status_std"]
    ==
    "ERROR"
)


# ------------------------------------------------------------
# SUCCESS sample_id boş olamaz
# ------------------------------------------------------------

empty_success_sample_ids = (
    success_roi_mask
    &
    roi["sample_id"].eq("")
)


if empty_success_sample_ids.any():

    bad_rows = roi.loc[
        empty_success_sample_ids,
        [
            "label",
            "split",
            "source_frame",
            "frame_stem",
            "face_id",
            "status_raw",
        ],
    ].head(20)


    raise AssertionError(
        "SUCCESS statülü Eye ROI kayıtlarında "
        "boş sample_id bulundu.\n\n"
        +
        bad_rows.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# SUCCESS sample_id unique olmalı
# ------------------------------------------------------------

success_sample_ids = (
    roi.loc[
        success_roi_mask,
        "sample_id",
    ]
)


duplicate_success_ids = (
    success_sample_ids[
        success_sample_ids.duplicated(
            keep=False
        )
    ]
)


if not duplicate_success_ids.empty:

    duplicate_values = set(
        duplicate_success_ids
    )


    duplicate_preview = (
        roi.loc[
            success_roi_mask
            &
            roi["sample_id"].isin(
                duplicate_values
            ),
            [
                "sample_id",
                "label",
                "split",
                "source_frame",
                "frame_stem",
                "face_id",
                "combined_eye_path",
            ],
        ]
        .head(30)
    )


    raise AssertionError(
        "SUCCESS Eye ROI kayıtlarında gerçek "
        "mükerrer sample_id tespit edildi.\n\n"
        +
        duplicate_preview.to_string(
            index=False
        )
    )


print(
    "[PASS] SUCCESS sample_id values are present and unique."
)


# ============================================================
# 6. CREATE TRACEABLE IDS FOR SKIPPED / ERROR ROWS
# ============================================================
#
# no_face satırlarına ROI üretimi sırasında sample_id verilmemiş.
#
# Metadata muhasebesinde her satırın benzersiz takip anahtarı
# olması için sadece BOŞ sample_id'lere deterministic audit ID
# oluşturuyoruz.
#
# Bu ID gerçek ROI sample_id değildir.
# ============================================================

def make_audit_sample_id(row_index, row):

    payload = (
        f"{row_index}|"
        f"{row['label']}|"
        f"{row['split']}|"
        f"{row['source_frame']}|"
        f"{row['frame_stem']}|"
        f"{row['status_raw']}"
    )

    digest = hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()[:16]

    return (
        f"audit_{row['status_raw']}_"
        f"{digest}"
    )


missing_sample_mask = (
    roi["sample_id"]
    .eq("")
)


generated_audit_ids = 0


for idx in roi.index[
    missing_sample_mask
]:

    roi.at[
        idx,
        "sample_id",
    ] = make_audit_sample_id(
        idx,
        roi.loc[idx],
    )

    generated_audit_ids += 1


print(
    f"Generated audit IDs for missing "
    f"sample_id rows: {generated_audit_ids}"
)


# ------------------------------------------------------------
# Now every metadata row must be uniquely traceable
# ------------------------------------------------------------

if roi["sample_id"].eq("").any():

    raise AssertionError(
        "Audit ID üretiminden sonra hâlâ "
        "boş sample_id bulunuyor."
    )


if not roi["sample_id"].is_unique:

    duplicate_all = (
        roi.loc[
            roi["sample_id"].duplicated(
                keep=False
            ),
            [
                "sample_id",
                "label",
                "split",
                "status_raw",
                "source_frame",
                "frame_stem",
            ],
        ]
        .head(30)
    )


    raise AssertionError(
        "Audit ID oluşturulduktan sonra "
        "metadata sample_id değerleri unique değil.\n\n"
        +
        duplicate_all.to_string(
            index=False
        )
    )


print(
    "[PASS] All metadata rows now have unique trace IDs."
)


# ============================================================
# 7. PATH RESOLUTION
# ============================================================

def resolve_combined_eye_path(
    value,
):

    if pd.isna(value):

        return ""


    value = str(
        value
    ).strip()


    if value == "":

        return ""


    p = Path(
        value
    )


    if p.is_absolute():

        return str(
            p
        )


    return str(
        DATA_ROOT
        /
        p
    )


# ============================================================
# 8. FRAME INDEX PARSER
# ============================================================

def parse_frame_index(
    frame_stem,
):

    match = re.search(
        r"(\d+)$",
        str(
            frame_stem
        ),
    )


    if match is None:

        return -1


    return int(
        match.group(1)
    )


# ============================================================
# 9. FACE INDEX PARSER
# ============================================================

def safe_face_index(
    value,
):

    if pd.isna(
        value
    ):

        return -1


    value_string = str(
        value
    ).strip()


    if value_string == "":

        return -1


    try:

        return int(
            float(
                value_string
            )
        )


    except Exception:

        return -1


# ============================================================
# 10. BUILD STANDARDIZED EXPERIMENT METADATA
# ============================================================
#
# video_id mevcut ROI metadata'dan korunur.
#
# Ancak bunu authoritative source-video identity olarak
# göstermiyoruz.
# ============================================================

metadata = pd.DataFrame(
    {

        "sample_id":
            roi["sample_id"],


        "sample_id_semantics":
            np.where(
                success_roi_mask,
                "roi_sample_id",
                "generated_audit_id",
            ),


        "source_video":
            roi["video_id"],


        "source_video_semantics":
            "roi_metadata_video_id",


        "source_frame":
            roi["source_frame"],


        "frame_index":
            (
                roi["frame_stem"]
                .map(
                    parse_frame_index
                )
                .astype(int)
            ),


        "face_index":
            (
                roi["face_id"]
                .map(
                    safe_face_index
                )
                .astype(int)
            ),


        "roi_state":
            "combined_eye",


        "label":
            roi["label"],


        "split":
            roi["split"],


        "status":
            roi["status_std"],


        "skip_reason":
            np.where(

                roi["status_std"]
                .eq(
                    "SKIPPED"
                ),

                roi["status_raw"],

                np.where(

                    roi["status_std"]
                    .eq(
                        "ERROR"
                    ),

                    (
                        roi["error"]
                        .fillna("")
                        .astype(str)
                        if "error" in roi.columns
                        else ""
                    ),

                    "",
                ),
            ),


        "sha256":
            "",


        "output_path":
            (
                roi["combined_eye_path"]
                .map(
                    resolve_combined_eye_path
                )
            ),


        "run_id":
            RUN_ID,


        "relative_frame_path":
            roi["relative_frame_path"],


        "frame_stem":
            roi["frame_stem"],


        "video_id":
            roi["video_id"],
    }
)


# ============================================================
# 11. OUTPUT PATH QUALITY GATE
# ============================================================

success_mask = (
    metadata["status"]
    .eq(
        "SUCCESS"
    )
)


if not metadata.loc[
    success_mask,
    "output_path",
].ne("").all():

    missing_paths = (
        metadata.loc[
            success_mask
            &
            metadata["output_path"].eq(""),
            [
                "sample_id",
                "label",
                "split",
                "source_frame",
                "frame_stem",
            ],
        ]
        .head(20)
    )


    raise AssertionError(
        "SUCCESS statülü bazı örneklerde "
        "combined_eye_path boş.\n\n"
        +
        missing_paths.to_string(
            index=False
        )
    )


# ============================================================
# 12. DATA ACCOUNTING
# ============================================================

total_inputs = int(
    len(
        metadata
    )
)


success_count = int(
    metadata["status"]
    .eq(
        "SUCCESS"
    )
    .sum()
)


skipped_count = int(
    metadata["status"]
    .eq(
        "SKIPPED"
    )
    .sum()
)


error_count = int(
    metadata["status"]
    .eq(
        "ERROR"
    )
    .sum()
)


assert (
    total_inputs
    ==
    success_count
    +
    skipped_count
    +
    error_count
), (
    "Girdi-çıktı sayıları uyuşmuyor."
)


print()
print("=" * 78)
print("EYE ROI DATA ACCOUNTING")
print("=" * 78)

print(
    f"Total   : {total_inputs:,}"
)

print(
    f"SUCCESS : {success_count:,}"
)

print(
    f"SKIPPED : {skipped_count:,}"
)

print(
    f"ERROR   : {error_count:,}"
)


# ============================================================
# 13. SUCCESS FILE EXISTENCE CHECK
# ============================================================

missing_success_files = [

    p

    for p

    in metadata.loc[
        success_mask,
        "output_path",
    ]

    if not Path(
        p
    ).exists()

]


if missing_success_files:

    raise FileNotFoundError(
        "SUCCESS statülü "
        f"{len(missing_success_files)} "
        "combined-eye dosyası bulunamadı.\n"
        "İlk örnekler:\n- "
        +
        "\n- ".join(
            missing_success_files[:10]
        )
    )


# ============================================================
# 14. TRAINING-ELIGIBLE DATA
# ============================================================

eligible = (
    metadata.loc[
        success_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if eligible.empty:

    raise AssertionError(
        "Eğitime uygun SUCCESS Eye ROI örneği yok."
    )


# ------------------------------------------------------------
# frame_index must exist for SUCCESS
# ------------------------------------------------------------

bad_frame_indices = (
    eligible["frame_index"]
    <
    0
)


if bad_frame_indices.any():

    preview = (
        eligible.loc[
            bad_frame_indices,
            [
                "sample_id",
                "source_frame",
                "frame_stem",
            ],
        ]
        .head(20)
    )


    raise AssertionError(
        "Bazı SUCCESS frame_index değerleri "
        "çözülemedi.\n\n"
        +
        preview.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# face_index must exist for SUCCESS
# ------------------------------------------------------------

bad_face_indices = (
    eligible["face_index"]
    <
    0
)


if bad_face_indices.any():

    preview = (
        eligible.loc[
            bad_face_indices,
            [
                "sample_id",
                "source_frame",
                "face_index",
            ],
        ]
        .head(20)
    )


    raise AssertionError(
        "Bazı SUCCESS face_index değerleri "
        "çözülemedi.\n\n"
        +
        preview.to_string(
            index=False
        )
    )


# ============================================================
# 15. SHA256 CONTENT HASH
# ============================================================
#
# Exact-content leakage kontrolü gerçek dosya içeriğine göre
# yapılır.
# ============================================================

logger.info(
    "SUCCESS Eye ROI dosyaları için "
    "SHA256 hesaplanıyor: %d örnek",
    len(
        eligible
    ),
)


hash_values = []


for i, path_string in enumerate(
    eligible["output_path"],
    start=1,
):

    hash_values.append(

        sha256_file(
            Path(
                path_string
            )
        )
    )


    if (
        i % 250 == 0
        or
        i == len(
            eligible
        )
    ):

        logger.info(
            "SHA256 progress: %d/%d",
            i,
            len(
                eligible
            ),
        )


eligible["sha256"] = (
    hash_values
)


# ------------------------------------------------------------
# Write hashes back using sample_id mapping rather than index
# ------------------------------------------------------------

sha_map = (
    eligible
    .set_index(
        "sample_id"
    )["sha256"]
    .to_dict()
)


metadata.loc[
    success_mask,
    "sha256",
] = (
    metadata.loc[
        success_mask,
        "sample_id",
    ]
    .map(
        sha_map
    )
)


# ============================================================
# 16. CROSS-SPLIT EXACT CONTENT LEAKAGE
# ============================================================

hash_split_counts = (
    eligible
    .groupby(
        "sha256"
    )["split"]
    .nunique()
)


cross_split_hashes = (
    hash_split_counts[
        hash_split_counts
        >
        1
    ]
)


cross_split_hash_count = int(
    len(
        cross_split_hashes
    )
)


if cross_split_hash_count > 0:

    bad_hash_values = set(
        cross_split_hashes.index
    )


    hash_preview = (
        eligible.loc[
            eligible["sha256"]
            .isin(
                bad_hash_values
            ),
            [
                "sample_id",
                "label",
                "split",
                "output_path",
                "sha256",
            ],
        ]
        .sort_values(
            [
                "sha256",
                "split",
            ]
        )
        .head(30)
    )


    raise AssertionError(
        "Exact aynı görüntü içeriği birden fazla "
        f"splitte bulundu: "
        f"{cross_split_hash_count} SHA256.\n\n"
        +
        hash_preview.to_string(
            index=False
        )
    )


print(
    "[PASS] Cross-split SHA256 duplicate check."
)


# ============================================================
# 17. CLASS PRESENCE CHECK
# ============================================================

for split_name in [
    "train",
    "val",
    "test",
]:

    part_labels = set(

        eligible.loc[
            eligible["split"]
            .eq(
                split_name
            ),
            "label",
        ]

    )


    if part_labels != {
        "real",
        "fake",
    }:

        raise AssertionError(
            f"{split_name} split iki sınıfı da "
            f"içermiyor: {sorted(part_labels)}"
        )


print(
    "[PASS] Both classes exist in Train/Val/Test."
)


# ============================================================
# 18. CROSS-SPLIT PATH LEAKAGE
# ============================================================

path_split_counts = (
    eligible
    .groupby(
        "output_path"
    )["split"]
    .nunique()
)


cross_split_path_count = int(
    (
        path_split_counts
        >
        1
    )
    .sum()
)


if cross_split_path_count > 0:

    raise AssertionError(
        "Aynı combined-eye dosya yolu "
        "birden fazla splitte bulundu: "
        f"{cross_split_path_count}"
    )


print(
    "[PASS] Cross-split output-path check."
)


# ============================================================
# 19. ROI METADATA video_id GROUP CHECK
# ============================================================
#
# Bunlar authoritative source-video ID olarak kabul edilmez.
# Yalnızca mevcut metadata grup alanı olarak denetlenir.
# ============================================================

group_sets = {

    split_name: set(

        eligible.loc[
            eligible["split"]
            .eq(
                split_name
            ),
            "video_id",
        ]
        .astype(str)
        .str.strip()

    )

    for split_name in [
        "train",
        "val",
        "test",
    ]
}


group_intersections = {

    "train_val":
        len(
            group_sets["train"]
            &
            group_sets["val"]
        ),

    "train_test":
        len(
            group_sets["train"]
            &
            group_sets["test"]
        ),

    "val_test":
        len(
            group_sets["val"]
            &
            group_sets["test"]
        ),
}


if not all(
    value == 0
    for value
    in group_intersections.values()
):

    raise AssertionError(
        "video_id/group leakage tespit edildi: "
        f"{group_intersections}"
    )


# Same current metadata group cannot cross split.
max_split_per_group = (
    eligible
    .groupby(
        "video_id"
    )["split"]
    .nunique()
    .max()
)


if max_split_per_group > 1:

    raise AssertionError(
        "Aynı ROI metadata video_id "
        "birden fazla split taşıyor."
    )


# Same current metadata group cannot cross labels.
max_label_per_group = (
    eligible
    .groupby(
        "video_id"
    )["label"]
    .nunique()
    .max()
)


if max_label_per_group > 1:

    raise AssertionError(
        "Aynı ROI metadata video_id "
        "birden fazla label taşıyor."
    )


print(
    "[PASS] Current ROI video_id group isolation."
)


# ============================================================
# 20. SPLIT COUNTS
# ============================================================

split_class_counts_df = (
    eligible
    .groupby(
        [
            "split",
            "label",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


split_class_counts = (
    split_class_counts_df
    .to_dict(
        orient="index"
    )
)


split_group_counts = {

    split_name: int(

        eligible.loc[
            eligible["split"]
            .eq(
                split_name
            ),
            "video_id",
        ]
        .nunique()

    )

    for split_name in [
        "train",
        "val",
        "test",
    ]
}


# ============================================================
# 21. TRUE VIDEO-LEVEL LEAKAGE STATUS
# ============================================================
#
# Current metadata does not prove original video identities.
# Therefore this quality gate is NOT falsely marked PASS.
# ============================================================

true_video_level_leakage_status = (
    "NOT_VERIFIABLE_FROM_CURRENT_METADATA"
)


# ============================================================
# 22. ACCOUNTING REPORT
# ============================================================

accounting = {

    "run_id":
        RUN_ID,


    "total_inputs":
        int(
            total_inputs
        ),


    "success_count":
        int(
            success_count
        ),


    "skipped_count":
        int(
            skipped_count
        ),


    "error_count":
        int(
            error_count
        ),


    "generated_audit_sample_ids":
        int(
            generated_audit_ids
        ),


    "training_eligible_success_count":
        int(
            len(
                eligible
            )
        ),


    "split_class_counts":
        split_class_counts,


    "video_id_group_counts":
        split_group_counts,


    "video_id_group_intersections":
        group_intersections,


    "cross_split_duplicate_output_path_count":
        int(
            cross_split_path_count
        ),


    "cross_split_duplicate_sha256_count":
        int(
            cross_split_hash_count
        ),


    "true_video_level_leakage_status":
        true_video_level_leakage_status,


    "true_video_level_note":
        (
            "ROI metadata video_id is preserved exactly "
            "as provided. In the current dataset it may "
            "represent split-level groups such as "
            "real_train/fake_test, not authoritative "
            "original source-video identity."
        ),
}


# ============================================================
# 23. SAVE OUTPUTS
# ============================================================

write_json_atomic(

    accounting,

    DIRS["metrics"]
    /
    "data_accounting.json",
)


write_csv_atomic(

    metadata,

    DIRS["artifacts"]
    /
    "metadata_used.csv",
)


# ============================================================
# 24. FINAL CELL OUTPUT
# ============================================================

print()
print("=" * 78)
print("EYE ROI METADATA QUALITY GATES COMPLETED")
print("=" * 78)

print(
    json.dumps(
        accounting,
        ensure_ascii=False,
        indent=2,
    )
)

[PASS] SUCCESS sample_id values are present and unique.
Generated audit IDs for missing sample_id rows: 111
[PASS] All metadata rows now have unique trace IDs.

EYE ROI DATA ACCOUNTING
Total   : 3,097
SUCCESS : 2,986
SKIPPED : 111
ERROR   : 0
2026-08-07 11:22:51,683 | INFO | SUCCESS Eye ROI dosyaları için SHA256 hesaplanıyor: 2986 örnek


INFO:eye_hog_lbp_svm:SUCCESS Eye ROI dosyaları için SHA256 hesaplanıyor: 2986 örnek


2026-08-07 11:26:30,561 | INFO | SHA256 progress: 250/2986


INFO:eye_hog_lbp_svm:SHA256 progress: 250/2986


2026-08-07 11:30:11,306 | INFO | SHA256 progress: 500/2986


...

2026-08-07 12:03:18,574 | INFO | SHA256 progress: 2750/2986


INFO:eye_hog_lbp_svm:SHA256 progress: 2750/2986


2026-08-07 12:06:45,995 | INFO | SHA256 progress: 2986/2986


INFO:eye_hog_lbp_svm:SHA256 progress: 2986/2986


[PASS] Cross-split SHA256 duplicate check.
[PASS] Both classes exist in Train/Val/Test.
[PASS] Cross-split output-path check.
[PASS] Current ROI video_id group isolation.

EYE ROI METADATA QUALITY GATES COMPLETED
{
  "run_id": "20260807_1119_eye_hog_lbp_svm_seed42",
  "total_inputs": 3097,
  "success_count": 2986,
  "skipped_count": 111,
  "error_count": 0,
  "generated_audit_sample_ids": 111,
  "training_eligible_success_count": 2986,
  "split_class_counts": {
    "test": {
      "fake": 156,
      "real": 146
    },
    "train": {
      "fake": 1191,
      "real": 1197
    },
    "val": {
      "fake": 141,
      "real": 155
    }
  },
  "video_id_group_counts": {
    "train": 2,
    "val": 2,
    "test": 2
  },
  "video_id_group_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "cross_split_duplicate_output_path_count": 0,
  "cross_split_duplicate_sha256_count": 0,
  "true_video_level_leakage_status": "NOT_VERIFIABLE_FROM_CURRENT_METADATA",
  "true_

In [7]:

# 5) HOG + LBP feature extraction fonksiyonları ve klasik-ML smoke test
IMG_W = int(CONFIG['image_width'])
IMG_H = int(CONFIG['image_height'])
HOG_CFG = CONFIG['hog']
LBP_CFG = CONFIG['lbp']


def load_preprocessed_grayscale(path: str) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Image could not be read: {path}')
    image = cv2.resize(image, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    return image


def extract_hog(gray: np.ndarray) -> np.ndarray:
    feat = hog(
        gray,
        orientations=int(HOG_CFG['orientations']),
        pixels_per_cell=tuple(HOG_CFG['pixels_per_cell']),
        cells_per_block=tuple(HOG_CFG['cells_per_block']),
        block_norm=str(HOG_CFG['block_norm']),
        feature_vector=True,
    )
    return np.asarray(feat, dtype=np.float32)


def extract_lbp(gray: np.ndarray) -> np.ndarray:
    points = int(LBP_CFG['points'])
    radius = int(LBP_CFG['radius'])
    method = str(LBP_CFG['method'])
    lbp_img = local_binary_pattern(gray, P=points, R=radius, method=method)
    # uniform LBP için P + 2 olası bin
    n_bins = points + 2 if method == 'uniform' else int(lbp_img.max() + 1)
    hist, _ = np.histogram(lbp_img.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
    hist = hist.astype(np.float32)
    hist /= hist.sum() + 1e-12
    return hist


def extract_features(path: str):
    gray = load_preprocessed_grayscale(path)
    hog_feat = extract_hog(gray)
    lbp_feat = extract_lbp(gray)
    fused = np.concatenate([hog_feat, lbp_feat]).astype(np.float32)
    if not np.isfinite(fused).all():
        raise FloatingPointError(f'NaN/Inf feature detected: {path}')
    return hog_feat, lbp_feat, fused


# Smoke gate: train setinden iki sınıfı içeren en az 4 örnekte extraction + fit + inference.
smoke_rows = pd.concat([
    eligible[(eligible['split'] == 'train') & (eligible['label'] == 'real')].head(2),
    eligible[(eligible['split'] == 'train') & (eligible['label'] == 'fake')].head(2),
], ignore_index=True)
assert len(smoke_rows) == 4 and smoke_rows['label'].nunique() == 2

smoke_X = []
smoke_y = []
for row in smoke_rows.itertuples(index=False):
    _, _, feat = extract_features(row.output_path)
    smoke_X.append(feat)
    smoke_y.append(1 if row.label == CONFIG['positive_class'] else 0)
smoke_X = np.vstack(smoke_X)
smoke_y = np.asarray(smoke_y)
smoke_scaler = StandardScaler().fit(smoke_X)
smoke_model = SVC(kernel='linear', C=1.0, class_weight='balanced')
smoke_model.fit(smoke_scaler.transform(smoke_X), smoke_y)
smoke_pred = smoke_model.predict(smoke_scaler.transform(smoke_X))
assert len(smoke_pred) == len(smoke_y)
assert np.isfinite(smoke_X).all()

print('Classical-ML smoke test: PASS')
print('Feature dimension:', smoke_X.shape[1])


Classical-ML smoke test: PASS
Feature dimension: 3790


In [8]:

# 6) Tüm splitler için feature extraction (split bazlı atomik cache)

def build_split_features(split: str):
    cache_path = DIRS['artifacts'] / f'features_{split}.npz'
    if cache_path.exists():
        logger.info('Cached features loaded: %s', cache_path)
        with np.load(cache_path, allow_pickle=False) as z:
            return {
                'X_hog': z['X_hog'],
                'X_lbp': z['X_lbp'],
                'X_fused': z['X_fused'],
                'y': z['y'],
                'sample_id': z['sample_id'].astype(str),
                'source_video': z['source_video'].astype(str),
                'frame_index': z['frame_index'],
                'output_path': z['output_path'].astype(str),
            }

    part = eligible[eligible['split'] == split].copy().reset_index(drop=True)
    hog_rows, lbp_rows, fused_rows = [], [], []
    labels = []

    start = time.time()
    for i, row in enumerate(part.itertuples(index=False), start=1):
        h, l, f = extract_features(row.output_path)
        hog_rows.append(h)
        lbp_rows.append(l)
        fused_rows.append(f)
        labels.append(1 if row.label == CONFIG['positive_class'] else 0)
        if i % 100 == 0 or i == len(part):
            logger.info('Feature extraction %s: %d/%d', split, i, len(part))

    data = {
        'X_hog': np.vstack(hog_rows).astype(np.float32),
        'X_lbp': np.vstack(lbp_rows).astype(np.float32),
        'X_fused': np.vstack(fused_rows).astype(np.float32),
        'y': np.asarray(labels, dtype=np.int64),
        'sample_id': part['sample_id'].astype(str).to_numpy(dtype=str),
        'source_video': part['source_video'].astype(str).to_numpy(dtype=str),
        'frame_index': part['frame_index'].to_numpy(dtype=np.int64),
        'output_path': part['output_path'].astype(str).to_numpy(dtype=str),
    }

    assert len(data['X_fused']) == len(part)
    assert np.isfinite(data['X_fused']).all()
    save_npz_atomic(cache_path, **data)
    logger.info('Feature extraction %s completed in %.2fs', split, time.time() - start)
    return data


features = {split: build_split_features(split) for split in ['train', 'val', 'test']}

hog_dim = int(features['train']['X_hog'].shape[1])
lbp_dim = int(features['train']['X_lbp'].shape[1])
fused_dim = int(features['train']['X_fused'].shape[1])
assert fused_dim == hog_dim + lbp_dim
for split in ['train', 'val', 'test']:
    assert features[split]['X_hog'].shape[1] == hog_dim
    assert features[split]['X_lbp'].shape[1] == lbp_dim
    assert features[split]['X_fused'].shape[1] == fused_dim

feature_dimensions = {
    'hog_dim': hog_dim,
    'lbp_dim': lbp_dim,
    'fused_dim': fused_dim,
    'image_size': [IMG_W, IMG_H],
}
write_json_atomic(feature_dimensions, DIRS['artifacts'] / 'feature_dimensions.json')

feature_pipeline = {
    'input': 'combined eye ROI',
    'preprocessing': ['grayscale', f'resize_{IMG_W}x{IMG_H}'],
    'features': {
        'HOG': HOG_CFG,
        'LBP': LBP_CFG,
        'fusion': 'concatenation',
    },
    'normalization': 'StandardScaler fit on train only',
    'classifier': 'SVM selected on validation only',
    'pretrained_model_used': False,
}
write_json_atomic(feature_pipeline, DIRS['artifacts'] / 'feature_pipeline.json')

print(json.dumps(feature_dimensions, indent=2))


2026-08-07 12:06:54,217 | INFO | Feature extraction train: 100/2388


INFO:eye_hog_lbp_svm:Feature extraction train: 100/2388


2026-08-07 12:06:57,473 | INFO | Feature extraction train: 200/2388


INFO:eye_hog_lbp_svm:Feature extraction train: 200/2388


2026-08-07 12:07:01,369 | INFO | Feature extraction train: 300/2388


...

2026-08-07 12:07:34,005 | INFO | Feature extraction test: 302/302


INFO:eye_hog_lbp_svm:Feature extraction test: 302/302


2026-08-07 12:07:34,257 | INFO | Feature extraction test completed in 3.13s


INFO:eye_hog_lbp_svm:Feature extraction test completed in 3.13s


{
  "hog_dim": 3780,
  "lbp_dim": 10,
  "fused_dim": 3790,
  "image_size": [
    128,
    64
  ]
}


In [9]:

# 7) Train-only scaling + validation-only SVM model/threshold selection
X_train = features['train']['X_fused']
y_train = features['train']['y']
X_val = features['val']['X_fused']
y_val = features['val']['y']
X_test = features['test']['X_fused']
y_test = features['test']['y']

# Normalizasyon yalnızca train üzerinde öğrenilir.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
# Test seti bu aşamada dönüştürülmez/değerlendirilmez.

atomic_joblib_dump(scaler, DIRS['artifacts'] / 'feature_preprocessing.joblib')


def metrics_from_scores(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else float('nan')
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'specificity': specificity,
        'roc_auc': roc_auc_score(y_true, scores),
        'average_precision': average_precision_score(y_true, scores),
    }


def best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        return 0.0
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    idx = int(np.nanargmax(f1_values))
    return float(thresholds[idx])


svm_cfg = CONFIG['svm']
candidates = []
for kernel in svm_cfg['kernels']:
    for C in svm_cfg['c_values']:
        if kernel == 'linear':
            candidates.append({'kernel': 'linear', 'C': float(C), 'gamma': None})
        elif kernel == 'rbf':
            for gamma in svm_cfg['gamma_values']:
                candidates.append({'kernel': 'rbf', 'C': float(C), 'gamma': gamma})
        else:
            raise ValueError(f'Unsupported kernel: {kernel}')

search_rows = []
search_start = time.time()
for idx, params in enumerate(candidates, start=1):
    kwargs = {
        'kernel': params['kernel'],
        'C': params['C'],
        'class_weight': svm_cfg['class_weight'],
        'probability': False,
        'cache_size': 2048,
    }
    if params['kernel'] == 'rbf':
        kwargs['gamma'] = params['gamma']

    model = SVC(**kwargs)
    t0 = time.time()
    model.fit(X_train_s, y_train)
    val_scores = model.decision_function(X_val_s)
    assert np.isfinite(val_scores).all(), 'Validation decision scores contain NaN/Inf.'
    threshold = best_f1_threshold(y_val, val_scores)
    m = metrics_from_scores(y_val, val_scores, threshold)
    row = {
        **params,
        'threshold': threshold,
        **{f'val_{k}': float(v) for k, v in m.items()},
        'fit_seconds': time.time() - t0,
    }
    search_rows.append(row)
    write_csv_atomic(pd.DataFrame(search_rows), DIRS['metrics'] / 'svm_validation_search.csv')
    logger.info('SVM candidate %d/%d: %s | val_f1=%.4f', idx, len(candidates), params, m['f1'])

search_df = pd.DataFrame(search_rows)
# Birincil seçim: val F1. Eşitlikte ROC-AUC, sonra accuracy.
search_df = search_df.sort_values(
    ['val_f1', 'val_roc_auc', 'val_accuracy'],
    ascending=[False, False, False],
).reset_index(drop=True)
best_row = search_df.iloc[0].to_dict()

best_kwargs = {
    'kernel': best_row['kernel'],
    'C': float(best_row['C']),
    'class_weight': svm_cfg['class_weight'],
    'probability': False,
    'cache_size': 2048,
}
if best_row['kernel'] == 'rbf':
    best_kwargs['gamma'] = best_row['gamma']

# Seçim validation ile tamamlandıktan sonra nihai model SADECE train üzerinde aynı parametrelerle fit edilir.
best_model = SVC(**best_kwargs)
best_model.fit(X_train_s, y_train)
selected_threshold = float(best_row['threshold'])

selection_summary = {
    'selection_set': 'validation',
    'selection_metric': svm_cfg['selection_metric'],
    'kernel': best_row['kernel'],
    'C': float(best_row['C']),
    'gamma': None if pd.isna(best_row.get('gamma')) else best_row.get('gamma'),
    'threshold': selected_threshold,
    'validation_metrics': {k.replace('val_', ''): float(v) for k, v in best_row.items() if str(k).startswith('val_')},
    'candidate_count': len(candidates),
    'search_seconds': time.time() - search_start,
}
write_json_atomic(selection_summary, DIRS['metrics'] / 'svm_selection.json')

# Standalone model artifact
model_artifact = {
    'model': best_model,
    'scaler': scaler,
    'threshold': selected_threshold,
    'positive_class': CONFIG['positive_class'],
    'feature_pipeline': feature_pipeline,
    'config': resolved_config,
}
atomic_joblib_dump(model_artifact, DIRS['artifacts'] / 'svm_model.joblib', required_keys=['model', 'scaler', 'threshold'])

print(json.dumps(json_safe(selection_summary), ensure_ascii=False, indent=2))


2026-08-07 12:08:12,353 | INFO | SVM candidate 1/12: {'kernel': 'linear', 'C': 0.1, 'gamma': None} | val_f1=0.6512


INFO:eye_hog_lbp_svm:SVM candidate 1/12: {'kernel': 'linear', 'C': 0.1, 'gamma': None} | val_f1=0.6512


2026-08-07 12:09:03,456 | INFO | SVM candidate 2/12: {'kernel': 'linear', 'C': 1.0, 'gamma': None} | val_f1=0.6453


INFO:eye_hog_lbp_svm:SVM candidate 2/12: {'kernel': 'linear', 'C': 1.0, 'gamma': None} | val_f1=0.6453


2026-08-07 12:14:29,839 | INFO | SVM candidate 3/12: {'kernel': 'linear', 'C': 10.0, 'gamma': None} | val_f1=0.6453


...

2026-08-07 12:16:39,278 | INFO | SVM candidate 11/12: {'kernel': 'rbf', 'C': 10.0, 'gamma': 0.001} | val_f1=0.6528


INFO:eye_hog_lbp_svm:SVM candidate 11/12: {'kernel': 'rbf', 'C': 10.0, 'gamma': 0.001} | val_f1=0.6528


2026-08-07 12:16:53,393 | INFO | SVM candidate 12/12: {'kernel': 'rbf', 'C': 10.0, 'gamma': 0.0001} | val_f1=0.6910


INFO:eye_hog_lbp_svm:SVM candidate 12/12: {'kernel': 'rbf', 'C': 10.0, 'gamma': 0.0001} | val_f1=0.6910


{
  "selection_set": "validation",
  "selection_metric": "f1",
  "kernel": "rbf",
  "C": 10.0,
  "gamma": 0.0001,
  "threshold": -0.5433359968892147,
  "validation_metrics": {
    "accuracy": 0.6283783783783784,
    "precision": 0.5720930232558139,
    "recall": 0.8723404255319149,
    "f1": 0.6910112359550562,
    "specificity": 0.4064516129032258,
    "roc_auc": 0.6794326241134752,
    "average_precision": 0.6484978664904018
  },
  "candidate_count": 12,
  "search_seconds": 557.0083870887756
}


In [10]:

# 8) Atomik klasik-ML checkpoint + fresh-load inference quality gates
checkpoint_state = {
    'checkpoint_schema': 'classical_ml_v1',
    'training_stage': 'completed_svm_fit',
    'epoch': None,
    'model': best_model,
    'preprocessor': scaler,
    'threshold': selected_threshold,
    'config': resolved_config,
    'best_metric_score': float(best_row['val_f1']),
    'python_random_state': random.getstate(),
    'numpy_random_state': np.random.get_state(),
    'non_applicable_pytorch_fields': {
        'model_state_dict': 'N/A - sklearn SVC object serialized as a whole',
        'optimizer_state_dict': 'N/A - SVC has no PyTorch optimizer',
        'scheduler_state_dict': 'N/A',
        'scaler_state_dict_mixed_precision': 'N/A',
        'forward_backward_smoke_test': 'N/A - classical ML; equivalent fit/predict smoke test used',
    },
}

for name in ['best.ckpt', 'last.ckpt', 'checkpoint_quality_gate.ckpt']:
    atomic_joblib_dump(
        checkpoint_state,
        DIRS['checkpoints'] / name,
        required_keys=['checkpoint_schema', 'model', 'preprocessor', 'threshold', 'config'],
    )

# Checkpoint reload: aynı validation skorlarını üretmeli.
reloaded_ckpt = joblib.load(DIRS['checkpoints'] / 'best.ckpt')
reload_val_scores = reloaded_ckpt['model'].decision_function(
    reloaded_ckpt['preprocessor'].transform(X_val)
)
original_val_scores = best_model.decision_function(X_val_s)
max_abs_diff = float(np.max(np.abs(reload_val_scores - original_val_scores)))
assert np.allclose(reload_val_scores, original_val_scores, rtol=1e-10, atol=1e-12)

# Fresh-load inference: bağımsız svm_model.joblib dosyası sıfırdan yüklenerek tahmin üretmeli.
fresh = joblib.load(DIRS['artifacts'] / 'svm_model.joblib')
probe_X = X_val[: min(8, len(X_val))]
probe_scores = fresh['model'].decision_function(fresh['scaler'].transform(probe_X))
probe_pred = (probe_scores >= fresh['threshold']).astype(int)
assert len(probe_pred) == len(probe_X)
assert np.isfinite(probe_scores).all()

fresh_load_test = {
    'status': 'PASS',
    'probe_count': len(probe_X),
    'predictions': probe_pred.tolist(),
    'checkpoint_reload_max_abs_score_diff': max_abs_diff,
}
write_json_atomic(fresh_load_test, DIRS['metrics'] / 'fresh_load_inference_test.json')

quality_gates = {
    'schema_test': 'PASS',
    'split_test': 'PASS_FOR_CURRENT_VIDEO_ID_GROUPS',
    'true_video_level_split_test': accounting['true_video_level_leakage_status'],
    'data_accounting_test': 'PASS',
    'exact_duplicate_sha256_split_test': 'PASS',
    'numerical_feature_test': 'PASS',
    'classical_ml_smoke_test': 'PASS',
    'pytorch_forward_backward_smoke_test': 'NOT_APPLICABLE_CLASSICAL_ML',
    'checkpoint_atomic_save_reload_test': 'PASS',
    'fresh_load_inference_test': 'PASS',
    'test_set_used_for_selection': False,
}
write_json_atomic(quality_gates, DIRS['metrics'] / 'quality_gates.json')
print(json.dumps(quality_gates, indent=2))


{
  "schema_test": "PASS",
  "split_test": "PASS_FOR_CURRENT_VIDEO_ID_GROUPS",
  "true_video_level_split_test": "NOT_VERIFIABLE_FROM_CURRENT_METADATA",
  "data_accounting_test": "PASS",
  "exact_duplicate_sha256_split_test": "PASS",
  "numerical_feature_test": "PASS",
  "classical_ml_smoke_test": "PASS",
  "pytorch_forward_backward_smoke_test": "NOT_APPLICABLE_CLASSICAL_ML",
  "checkpoint_atomic_save_reload_test": "PASS",
  "fresh_load_inference_test": "PASS",
  "test_set_used_for_selection": false
}


In [11]:

# 9) Nihai TEST değerlendirmesi — test setine ilk kez burada bakılır
# Bu hücreye kadar hiçbir test metriği model/threshold seçimi için kullanılmadı.

def build_prediction_df(split_name, split_features, scores, threshold):
    y_true = split_features['y']
    y_pred = (scores >= threshold).astype(int)
    return pd.DataFrame({
        'sample_id': split_features['sample_id'],
        'source_video': split_features['source_video'],
        'frame_index': split_features['frame_index'],
        'split': split_name,
        'y_true': y_true,
        'label': np.where(y_true == 1, CONFIG['positive_class'], 'real'),
        'decision_score': scores,
        'threshold': threshold,
        'y_pred': y_pred,
        'predicted_label': np.where(y_pred == 1, CONFIG['positive_class'], 'real'),
        'correct': y_true == y_pred,
        'output_path': split_features['output_path'],
        'run_id': RUN_ID,
    })


def aggregate_group_level(frame_df: pd.DataFrame):
    label_nunique = frame_df.groupby('source_video')['y_true'].nunique()
    assert label_nunique.max() == 1, 'A source_video has conflicting labels.'
    video = frame_df.groupby('source_video', as_index=False).agg(
        y_true=('y_true', 'first'),
        decision_score=('decision_score', 'mean'),
        frame_count=('sample_id', 'size'),
    )
    video['threshold'] = selected_threshold
    video['y_pred'] = (video['decision_score'] >= selected_threshold).astype(int)
    video['label'] = np.where(video['y_true'] == 1, CONFIG['positive_class'], 'real')
    video['predicted_label'] = np.where(video['y_pred'] == 1, CONFIG['positive_class'], 'real')
    video['correct'] = video['y_true'] == video['y_pred']
    video['run_id'] = RUN_ID
    return video


X_test_s = scaler.transform(X_test)
test_scores = best_model.decision_function(X_test_s)
assert np.isfinite(test_scores).all()

frame_pred = build_prediction_df('test', features['test'], test_scores, selected_threshold)
video_pred = aggregate_group_level(frame_pred)

frame_metrics = metrics_from_scores(frame_pred['y_true'].to_numpy(), frame_pred['decision_score'].to_numpy(), selected_threshold)
video_metrics = metrics_from_scores(video_pred['y_true'].to_numpy(), video_pred['decision_score'].to_numpy(), selected_threshold)

write_csv_atomic(frame_pred, DIRS['predictions'] / 'test_predictions_frame_level.csv')
write_csv_atomic(video_pred, DIRS['predictions'] / 'test_predictions_group_level.csv')
write_json_atomic(frame_metrics, DIRS['metrics'] / 'test_metrics_frame_level.json')
write_json_atomic(video_metrics, DIRS['metrics'] / 'test_metrics_group_level.json')

metrics_summary = pd.DataFrame([
    {'level': 'frame', **frame_metrics},
    {'level': 'group_metadata_video_id', **video_metrics},
])
write_csv_atomic(metrics_summary, DIRS['metrics'] / 'test_metrics_summary.csv')

print(metrics_summary.to_string(index=False))


                  level  accuracy  precision   recall       f1  specificity  roc_auc  average_precision
                  frame  0.592715   0.571429 0.846154 0.682171     0.321918 0.603223           0.605107
group_metadata_video_id  0.500000   0.500000 1.000000 0.666667     0.000000 1.000000           1.000000


In [12]:

# 10) İngilizce ve yüksek çözünürlüklü rapor grafikleri
figure_records = []


def register_figure(fig, filename):
    path = DIRS['figures'] / filename
    width, height = save_figure(fig, path)
    figure_records.append({
        'file': filename,
        'width_px': width,
        'height_px': height,
        'short_edge_px': min(width, height),
        'quality_pass': min(width, height) >= int(CONFIG['minimum_figure_short_edge_px']),
    })


# Dataset distribution
count_df = eligible.groupby(['split', 'label']).size().unstack(fill_value=0).reindex(['train', 'val', 'test'])
fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
x = np.arange(len(count_df.index))
width = 0.35
ax.bar(x - width/2, count_df.get('real', pd.Series(0, index=count_df.index)), width, label='Real')
ax.bar(x + width/2, count_df.get('fake', pd.Series(0, index=count_df.index)), width, label='Fake')
ax.set_title('Dataset Distribution by Split and Class', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Split', fontsize=11)
ax.set_ylabel('Number of Frames', fontsize=11)
ax.set_xticks(x, ['Train', 'Validation', 'Test'])
ax.legend(frameon=True)
ax.grid(True, axis='y', alpha=0.25)
register_figure(fig, 'dataset_distribution.png')

# Validation candidate comparison
plot_search = search_df.copy().sort_values('val_f1', ascending=False).reset_index(drop=True)
labels = [
    f"{r.kernel} | C={r.C}" + (f" | g={r.gamma}" if r.kernel == 'rbf' else '')
    for r in plot_search.itertuples(index=False)
]
fig, ax = plt.subplots(figsize=(12, 7), dpi=int(CONFIG['figure_dpi']))
ax.bar(np.arange(len(plot_search)), plot_search['val_f1'])
ax.set_title('Validation F1 by SVM Candidate', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('SVM Candidate', fontsize=11)
ax.set_ylabel('Validation F1', fontsize=11)
ax.set_xticks(np.arange(len(labels)), labels, rotation=60, ha='right')
ax.grid(True, axis='y', alpha=0.25)
register_figure(fig, 'svm_validation_f1_comparison.png')


def confusion_plot(df, level_name, filename):
    cm = confusion_matrix(df['y_true'], df['y_pred'], labels=[0, 1])
    fig, ax = plt.subplots(figsize=(8, 8), dpi=int(CONFIG['figure_dpi']))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Real', 'Fake'])
    disp.plot(ax=ax, colorbar=False, values_format='d')
    ax.set_title(f'Test Confusion Matrix — {level_name}', fontsize=14, fontweight='bold', pad=12)
    register_figure(fig, filename)


def roc_plot(df, level_name, filename):
    fpr, tpr, _ = roc_curve(df['y_true'], df['decision_score'])
    auc_value = roc_auc_score(df['y_true'], df['decision_score'])
    fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
    ax.plot(fpr, tpr, linewidth=2, label=f'ROC AUC = {auc_value:.3f}')
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
    ax.set_title(f'Test ROC Curve — {level_name}', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.legend(frameon=True, loc='lower right')
    ax.grid(True, alpha=0.25)
    register_figure(fig, filename)


def pr_plot(df, level_name, filename):
    precision, recall, _ = precision_recall_curve(df['y_true'], df['decision_score'])
    ap = average_precision_score(df['y_true'], df['decision_score'])
    fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
    ax.plot(recall, precision, linewidth=2, label=f'Average Precision = {ap:.3f}')
    ax.set_title(f'Test Precision-Recall Curve — {level_name}', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Recall', fontsize=11)
    ax.set_ylabel('Precision', fontsize=11)
    ax.legend(frameon=True, loc='lower left')
    ax.grid(True, alpha=0.25)
    register_figure(fig, filename)


confusion_plot(frame_pred, 'Frame Level', 'test_confusion_matrix_frame_level.png')
confusion_plot(video_pred, 'Group Level (metadata video_id)', 'test_confusion_matrix_group_level.png')
roc_plot(frame_pred, 'Frame Level', 'test_roc_curve_frame_level.png')
roc_plot(video_pred, 'Group Level (metadata video_id)', 'test_roc_curve_group_level.png')
pr_plot(frame_pred, 'Frame Level', 'test_precision_recall_curve_frame_level.png')
pr_plot(video_pred, 'Group Level (metadata video_id)', 'test_precision_recall_curve_group_level.png')

# Decision score distribution
fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
real_scores = frame_pred.loc[frame_pred['y_true'] == 0, 'decision_score']
fake_scores = frame_pred.loc[frame_pred['y_true'] == 1, 'decision_score']
ax.hist(real_scores, bins=30, alpha=0.65, label='Real')
ax.hist(fake_scores, bins=30, alpha=0.65, label='Fake')
ax.axvline(selected_threshold, linestyle='--', linewidth=2, label=f'Threshold = {selected_threshold:.3f}')
ax.set_title('Test Decision Score Distribution — Frame Level', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('SVM Decision Score', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(frameon=True)
ax.grid(True, alpha=0.25)
register_figure(fig, 'test_decision_score_distribution.png')

figure_audit = pd.DataFrame(figure_records)
assert figure_audit['quality_pass'].all(), 'At least one figure violates the minimum resolution rule.'
write_csv_atomic(figure_audit, DIRS['metrics'] / 'figure_quality_audit.csv')
figure_audit


,file,width_px,height_px,short_edge_px,quality_pass
0,dataset_distribution.png,1485,885,885,True
1,svm_validation_f1_comparison.png,1785,1035,1035,True
2,test_confusion_matrix_frame_level.png,1165,1185,1165,True
3,test_confusion_matrix_group_level.png,1165,1185,1165,True
4,test_roc_curve_frame_level.png,1485,885,885,True
5,test_roc_curve_group_level.png,1485,885,885,True
6,test_precision_recall_curve_frame_level.png,1485,885,885,True
7,test_precision_recall_curve_group_level.png,1485,885,885,True
8,test_decision_score_distribution.png,1485,885,885,True


In [13]:

# 11) Environment lock, output manifest ve run summary
# Gerçek Colab ortamı deney sonunda kilitlenir.
requirements_text = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'requirements_lock.txt').write_text(requirements_text, encoding='utf-8')

environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': __import__('sklearn').__version__,
    'scikit_image': __import__('skimage').__version__,
    'run_id': RUN_ID,
}
write_json_atomic(environment, RUN_DIR / 'environment.json')

run_summary = {
    'run_id': RUN_ID,
    'status': 'COMPLETED',
    'model': {
        'pretrained_weights': False,
        'feature_fusion': ['HOG', 'LBP'],
        'roi': 'combined_eye',
        'classifier': f"{selection_summary['kernel'].upper()} SVM",
        'C': selection_summary['C'],
        'gamma': selection_summary['gamma'],
        'threshold': selection_summary['threshold'],
    },
    'paths': {
        'input_data': str(DATA_ROOT),
        'roi_metadata': str(ROI_METADATA_PATH),
        'output_run': str(RUN_DIR),
        'best_checkpoint': str(DIRS['checkpoints'] / 'best.ckpt'),
        'svm_model': str(DIRS['artifacts'] / 'svm_model.joblib'),
    },
    'data': accounting,
    'feature_dimensions': feature_dimensions,
    'selected_svm': selection_summary,
    'test_metrics': {
        'frame_level': frame_metrics,
        'group_level_from_metadata_video_id': video_metrics,
    },
    'quality_gates': quality_gates,
    'figure_count': len(figure_records),
    'checkpoint_note': (
        'PyTorch optimizer/epoch state is not applicable to HOG+LBP+sklearn SVC. '
        'The complete SVC, train-fitted StandardScaler, threshold, config and RNG states '
        'are atomically serialized and reload-tested.'
    ),
    'completed_at': datetime.now().isoformat(),
}
write_json_atomic(run_summary, RUN_DIR / 'run_summary.json')

# Manifest kendisini hash'leyemez (self-referential), bu nedenle output_manifest.csv kendisini bilinçli olarak dışlar.
manifest_rows = []
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file() and path.name != 'output_manifest.csv' and not path.name.endswith('.tmp'):
        manifest_rows.append({
            'relative_path': str(path.relative_to(RUN_DIR)),
            'size_bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        })
manifest = pd.DataFrame(manifest_rows)
write_csv_atomic(manifest, RUN_DIR / 'output_manifest.csv')

# Son kontrol: geçici dosya kalmamalı.
tmp_files = [str(p) for p in RUN_DIR.rglob('*.tmp')]
assert not tmp_files, f'Temporary files remain: {tmp_files}'

print('=' * 80)
print('EXPERIMENT COMPLETED')
print('Run ID:', RUN_ID)
print('Output:', RUN_DIR)
print('\nFrame-level test metrics:')
print(json.dumps(json_safe(frame_metrics), indent=2))
print('\nGroup-level test metrics (metadata video_id):')
print(json.dumps(json_safe(video_metrics), indent=2))
print('\nGenerated output files:', len(manifest))
print('=' * 80)


EXPERIMENT COMPLETED
Run ID: 20260807_1119_eye_hog_lbp_svm_seed42
Output: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1119_eye_hog_lbp_svm_seed42

Frame-level test metrics:
{
  "accuracy": 0.5927152317880795,
  "precision": 0.5714285714285714,
  "recall": 0.8461538461538461,
  "f1": 0.6821705426356589,
  "specificity": 0.3219178082191781,
  "roc_auc": 0.6032226905514577,
  "average_precision": 0.6051072422628753
}

Group-level test metrics (metadata video_id):
{
  "accuracy": 0.5,
  "precision": 0.5,
  "recall": 1.0,
  "f1": 0.6666666666666666,
  "specificity": 0.0,
  "roc_auc": 1.0,
  "average_precision": 1.0
}

Generated output files: 36
